In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import pickle
import os

In [2]:
df = pd.read_csv('../data/healthcare-dataset-stroke-data.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Distribution:\n", df['stroke'].value_counts())
df.head()

Dataset Shape: (5110, 12)

Columns: ['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']

Missing Values:
 id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

Target Distribution:
 stroke
0    4861
1     249
Name: count, dtype: int64


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
df_clean = df.copy()

# Drop id
df_clean = df_clean.drop(columns=['id'])
print("Dropped 'id' column")

# Fill missing bmi with median
bmi_median = df_clean['bmi'].median()
df_clean['bmi'] = df_clean['bmi'].fillna(bmi_median)
print(f"Filled {df['bmi'].isnull().sum()} missing 'bmi' values with median: {bmi_median:.2f}")

print("\nMissing values remaining:", df_clean.isnull().sum().sum())
print("Shape:", df_clean.shape)

Dropped 'id' column
Filled 201 missing 'bmi' values with median: 28.10

Missing values remaining: 0
Shape: (5110, 11)


In [4]:
df_encoded = df_clean.copy()

binary_cols = ['gender', 'ever_married', 'Residence_type']
le = LabelEncoder()

for col in binary_cols:
    print(f"'{col}' unique values: {df_encoded[col].unique()}")
    df_encoded[col] = le.fit_transform(df_encoded[col])
    print(f"  Encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}\n")

multi_cols = ['work_type', 'smoking_status']
for col in multi_cols:
    dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=True)
    df_encoded = pd.concat([df_encoded.drop(columns=[col]), dummies], axis=1)
    print(f"One-hot encoded '{col}'")

print("\nShape after encoding:", df_encoded.shape)
print("\nColumns:", list(df_encoded.columns))

'gender' unique values: ['Male' 'Female' 'Other']
  Encoded: {'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2)}

'ever_married' unique values: ['Yes' 'No']
  Encoded: {'No': np.int64(0), 'Yes': np.int64(1)}

'Residence_type' unique values: ['Urban' 'Rural']
  Encoded: {'Rural': np.int64(0), 'Urban': np.int64(1)}

One-hot encoded 'work_type'
One-hot encoded 'smoking_status'

Shape after encoding: (5110, 16)

Columns: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'Residence_type', 'avg_glucose_level', 'bmi', 'stroke', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']


In [5]:
# 1. Initial 80/20 split (Stratified to keep stroke ratio)
X = df_encoded.drop(columns=['stroke'])
y = df_encoded['stroke']

X_train, X_test_full, y_train, y_test_full = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training class distribution:")
print(y_train.value_counts())
print(f"  Stroke ratio: {y_train.mean():.4f}")

# 2. Extract the "Dashboard 20" from the test set for your website demo
# Reset indices to avoid index issues
X_test_full_reset = X_test_full.reset_index(drop=True)
y_test_full_reset = y_test_full.reset_index(drop=True)
test_df = pd.concat([X_test_full_reset, y_test_full_reset], axis=1)

demo_stroke = test_df[test_df['stroke'] == 1].head(10)
demo_no_stroke = test_df[test_df['stroke'] == 0].head(10)

demo_set = pd.concat([demo_stroke, demo_no_stroke]).sample(frac=1, random_state=42)
demo_set.to_csv('../data/processed/dash_demo_records.csv', index=False)

# Remove the demo records from the final test set
demo_indices = demo_set.index
X_test = X_test_full_reset.drop(demo_indices)
y_test = y_test_full_reset.drop(demo_indices)

# No SMOTE - keeping original imbalanced data
# Class weights will be used in models instead

print(f"\nTraining Pool for CV: {len(X_train):,} rows (Imbalanced - using class weights)")
print(f"Final Test Set: {len(X_test):,} rows (Unseen)")
print(f"Demo Records Saved: {len(demo_set):,} rows")
print(f"Test set stroke ratio: {y_test.mean():.4f}")

Training class distribution:
stroke
0    3889
1     199
Name: count, dtype: int64
  Stroke ratio: 0.0487

Training Pool for CV: 4,088 rows (Imbalanced - using class weights)
Final Test Set: 1,002 rows (Unseen)
Demo Records Saved: 20 rows
Test set stroke ratio: 0.0399


In [6]:
numeric_cols = ['age', 'avg_glucose_level', 'bmi']
outlier_bounds = {}

for col in numeric_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_bounds[col] = (lower, upper)
    
    outliers = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    if outliers > 0:
        X_train[col] = X_train[col].clip(lower, upper)
        print(f"'{col}': capped {outliers} outliers in training set")

for col in numeric_cols:
    lower, upper = outlier_bounds[col]
   
    X_test[col] = X_test[col].clip(lower, upper)

print("\nOutlier handling complete.")

'avg_glucose_level': capped 503 outliers in training set
'bmi': capped 101 outliers in training set

Outlier handling complete.


In [7]:
scaler = StandardScaler()
feature_cols = X_train.columns.tolist()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete.")
print("Mean of scaled training features (should be ~0):",
      X_train_scaled[numeric_cols].mean().mean().round(4))

Scaling complete.
Mean of scaled training features (should be ~0): -0.0


In [8]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../artifacts', exist_ok=True)

# Unscaled (for tree models)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

# Scaled (for any distance-based models)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)

# Target labels
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler and feature list
with open('../artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("All files saved!")
for f in os.listdir('../data/processed'):
    print(f"  {f}")

All files saved!
  dash_demo_records.csv
  X_test.csv
  X_test_scaled.csv
  X_train.csv
  X_train_scaled.csv
  y_test.csv
  y_train.csv


In [9]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Original dataset shape:       {df.shape}")
print(f"After encoding shape:         {df_encoded.shape}")
print(f"Features used:                {len(feature_cols)}")
print(f"Target classes:               0 = No Stroke, 1 = Stroke")
print(f"\nSplit sizes (before balancing):")
print(f"  Train:      {len(X_train):,} rows (balanced)")
print(f"  Test:       {len(X_test):,} rows (original distribution)")
print("\nArtifacts saved: scaler.pkl, feature_columns.pkl")

PREPROCESSING SUMMARY
Original dataset shape:       (5110, 12)
After encoding shape:         (5110, 16)
Features used:                15
Target classes:               0 = No Stroke, 1 = Stroke

Split sizes (before balancing):
  Train:      4,088 rows (balanced)
  Test:       1,002 rows (original distribution)

Artifacts saved: scaler.pkl, feature_columns.pkl
